In [3]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
# ── Data ──────────────────────────────────────────────────────────────────────
# California Housing: 20,640 samples, 8 features → predict median house price.
# This is a regression target (continuous $), so MSE is selected as loss function.
data  = fetch_california_housing()
X, y  = data.data, data.target.reshape(-1, 1)   # y shape: (N,1) not (N,)

X = StandardScaler().fit_transform(X)            # zero mean, unit variance per feature
y = (y - y.mean()) / y.std()                     # normalise target too — keeps gradients small

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)



In [5]:
# ── Helpers ───────────────────────────────────────────────────────────────────
relu  = lambda z: np.maximum(0, z)
relu_ = lambda z: (z > 0).astype(float)          # derivative: 1 where active, 0 where dead



In [6]:
# MSE and its derivative w.r.t predictions
mse   = lambda p, t: np.mean((p - t) ** 2)
dmse  = lambda p, t: 2 * (p - t) / len(p)        # dL/d(y_pred) — feeds into backprop

In [7]:
# ── Architecture ──────────────────────────────────────────────────────────────
# Input(8) → Hidden(64, ReLU) → Output(1, linear)
# Linear output — no activation — because we're predicting a real number, not a probability.
# Sigmoid/tanh on output would squash predictions into (0,1) and destroy range.
np.random.seed(42)
H = 64
W1 = np.random.randn(8,  H)
b1 = np.zeros((1, H))
W2 = np.random.randn(H,  1)
b2 = np.zeros((1, 1))



In [8]:
# ── Training ──────────────────────────────────────────────────────────────────
lr, epochs = 0.01, 200

for epoch in range(epochs):

    # Forward: two affine transforms, one ReLU, one linear output
    z1   = X_tr @ W1 + b1                        # (N, H)
    a1   = relu(z1)                               # neurons with z1<0 output 0 — they're "dead" this pass
    pred = a1 @ W2 + b2                           # (N, 1) — raw real-valued prediction

    loss = mse(pred, y_tr)

    # Backward: chain rule from loss → W2 → W1
    # Each dW is: (upstream gradient) × (local gradient)
    dL   = dmse(pred, y_tr)                       # dL/d(pred): how loss changes with prediction
    dW2  = a1.T @ dL                              # dL/dW2: how prediction changes with W2
    db2  = dL.sum(axis=0, keepdims=True)

    da1  = dL @ W2.T                              # propagate error back through W2
    dz1  = da1 * relu_(z1)                        # ReLU gate: zero out gradient for dead neurons
    dW1  = X_tr.T @ dz1                           # dL/dW1
    db1  = dz1.sum(axis=0, keepdims=True)

    # Gradient descent step — subtract gradient scaled by lr
    W1 -= lr * dW1;  b1 -= lr * db1
    W2 -= lr * dW2;  b2 -= lr * db2

    if epoch % 20 == 0:
        test_loss = mse(relu(X_te @ W1 + b1) @ W2 + b2, y_te)
        print(f"Epoch {epoch:3d}  train MSE {loss:.4f}  test MSE {test_loss:.4f}")



Epoch   0  train MSE 167.1000  test MSE 326.5132
Epoch  20  train MSE 2.5293  test MSE 2.4266
Epoch  40  train MSE 1.5525  test MSE 1.5613
Epoch  60  train MSE 1.2195  test MSE 1.2544
Epoch  80  train MSE 1.0522  test MSE 1.0936
Epoch 100  train MSE 0.9451  test MSE 0.9879
Epoch 120  train MSE 0.8679  test MSE 0.9106
Epoch 140  train MSE 0.8082  test MSE 0.8505
Epoch 160  train MSE 0.7604  test MSE 0.8022
Epoch 180  train MSE 0.7213  test MSE 0.7624


$$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \overline{y})^2}$$


In [10]:
# ── Evaluation ────────────────────────────────────────────────────────────────
# R² tells us what fraction of variance the model explains (1.0 = perfect)
y_hat = relu(X_te @ W1 + b1) @ W2 + b2
ss_res = np.sum((y_te - y_hat) ** 2)
ss_tot = np.sum((y_te - y_te.mean()) ** 2)
print(f"\nR²: {1 - ss_res/ss_tot:.4f}")


R²: 0.2577
